# `07_RAG1.ipynb`

```sh
uv add "langchain[openai]" langchain-text-splitters requests numpy deepagents
```

## Store (저장)

In [1]:
from dotenv import load_dotenv
load_dotenv()

DOCS_BASE = "https://docs.langchain.com"

# Curated LangChain OSS pages for this tutorial. Expand this list or parse
# URLs from https://docs.langchain.com/llms.txt to index more of the site.
DOC_PATHS = [
    "/oss/python/langchain/agents",
    "/oss/python/deepagents/rag",
    "/oss/python/langchain/tools",
    "/oss/python/langchain/models",
    "/oss/python/deepagents/retrieval",
    "/oss/python/langchain/knowledge-base",
    "/oss/python/langchain/middleware",
    "/oss/python/deepagents/overview",
    "/oss/python/deepagents/subagents",
    "/oss/python/deepagents/streaming",
    "/oss/python/deepagents/frontend/subagent-streaming",
    "/oss/python/deepagents/backends",
    "/oss/python/langgraph/overview",
    "/oss/python/langgraph/quickstart",
]

In [2]:
# 1. Load (PDF, HTML, TEXT, MD, IMG, VIDEO, HWPX, XLSX, PPTX, DOCX) -> 문서 종류에 따라 방법이 다름

import requests
from langchain_core.documents import Document  # RAG에 사용할 문서 쪼가리를 의미하는 데이터 타입

# Node, Edge 이런거 아님. 단순 함수
def load_langchain_docs():
    docs = []

    for path in DOC_PATHS:
        url = f'{DOCS_BASE}{path}.md'
        res = requests.get(url, timeout=5)  # 5초간 답이 없으면 넘어가라
        # 단순 str 말고 Document 타입으로 잘 감싸기 -> RAG에 사용하기 위해
        doc = Document(page_content=res.text, metadata={'source': f'{DOCS_BASE}{path}'})
        docs.append(doc)

    return docs

docs = load_langchain_docs()
print(f'{len(docs)}개의 문서를 불러왔습니다')

14개의 문서를 불러왔습니다


In [3]:
# 2. Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = splitter.split_documents(docs)
print(f'{len(splits)}개의 조각으로 잘랐습니다')

990개의 조각으로 잘랐습니다


In [4]:
# 3. Embed
from langchain_openai import OpenAIEmbeddings

# embedding 담당자
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [6]:
# 4. Store
from langchain_core.vectorstores import InMemoryVectorStore

# 벡터스토어 세팅(임베딩)
vectorstore = InMemoryVectorStore(embedding=embeddings)

# 벡터스토어 저장 (문서조각)
vectorstore.add_documents(documents=splits)  # 리턴값은 저장된 문서 ID (쓸모없음)
print('저장완료')

저장완료


## Retrieve (검색)

In [8]:
from pprint import pprint

for doc in vectorstore.similarity_search('RAG하는법'):
    pprint(doc.page_content)

('## RAG architectures\n'
 '\n'
 "RAG can be implemented in multiple ways, depending on your system's needs. "
 'We outline each type in the sections below.')
('For more on RAG:\n'
 '\n'
 '* [Retrieval overview](/oss/python/deepagents/retrieval)\n'
 '* [RAG with Deep Agents](/oss/python/deepagents/rag)\n'
 '* [Evaluate a RAG application](/langsmith/evaluate-rag-tutorial)\n'
 '\n'
 '***\n'
 '\n'
 '<div className="source-links">\n'
 '  <Callout icon="terminal-2">\n'
 '    [Connect these docs](/use-these-docs) to Claude, VSCode, and more via '
 'MCP for real-time answers.\n'
 '  </Callout>\n'
 '\n'
 '  <Callout icon="edit">\n'
 '    [Edit this page on '
 'GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/langchain/knowledge-base.mdx) '
 'or [file an issue](https://github.com/langchain-ai/docs/issues/new/choose).\n'
 '  </Callout>\n'
 '</div>')
('### Hybrid RAG\n'
 '\n'
 'Hybrid RAG combines characteristics of both 2-Step and Agentic RAG. It '
 'introduces intermediate steps s